In [47]:
import numpy as np
import pandas as pd
import pickle
import joblib

In [10]:
#Load the dataset
df = pd.read_csv('final_dataset.csv')


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [12]:
# Encode categorical values using Label Encoding
encoder = LabelEncoder()
df['batting_team'] = encoder.fit_transform(df['batting_team'])
df['bowling_team'] = encoder.fit_transform(df['bowling_team'])
df['venue_canonical'] = encoder.fit_transform(df['venue_canonical'])

In [13]:
# Initialize LabelEncoders
batting_encoder = LabelEncoder()
bowling_encoder = LabelEncoder()
venue_encoder = LabelEncoder()


In [14]:
# Fit and transform categorical columns
df['batting_team'] = batting_encoder.fit_transform(df['batting_team'])
df['bowling_team'] = bowling_encoder.fit_transform(df['bowling_team'])
df['venue_canonical'] = venue_encoder.fit_transform(df['venue_canonical'])


In [15]:
# Save the LabelEncoders
joblib.dump(batting_encoder, 'batting_encoder.pkl')
joblib.dump(bowling_encoder, 'bowling_encoder.pkl')
joblib.dump(venue_encoder, 'venue_encoder.pkl')

print("Label Encoders Saved Successfully!")

Label Encoders Saved Successfully!


In [16]:
# Create binary target variable (win)
df['win'] = (df['win'] == 1).astype(int)

In [17]:
#  Train-Test Split
X = df.drop(columns=['win'])  # Features
y = df['win']                # Target

# Split data into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display shapes of train and test sets
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(51155, 10) (12789, 10) (51155,) (12789,)


In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# **Logistic Regression**

In [19]:
#Train Baseline Models
# Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [20]:
#Generate Predictions
y_train_pred = log_reg.predict(X_train)  # Predictions for training data
y_test_pred = log_reg.predict(X_test)    # Predictions for testing data


In [21]:
#Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

In [22]:
#Display Results
print(f"Training Accuracy: {train_accuracy:.4f}\n")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}\n")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

Training Accuracy: 0.6927

Training Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.62      0.66     24329
           1       0.69      0.76      0.72     26826

    accuracy                           0.69     51155
   macro avg       0.69      0.69      0.69     51155
weighted avg       0.69      0.69      0.69     51155

Training Confusion Matrix:
 [[15001  9328]
 [ 6391 20435]]

Testing Accuracy: 0.6936

Testing Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.62      0.66      6059
           1       0.69      0.76      0.72      6730

    accuracy                           0.69     12789
   macro avg       0.69      0.69      0.69     12789
weighted avg       0.69      0.69      0.69     12789

Testing Confusion Matrix:
 [[3728 2331]
 [1587 5143]]


In [23]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

In [24]:
# Define hyperparameter distribution
param_dist = {
    'C': uniform(0.01, 10),  # Random values between 0.01 and 10
    'solver': ['liblinear', 'saga', 'newton-cg', 'lbfgs'],  # Different solvers
    'max_iter': [1000, 2000, 3000],  # Number of iterations
    'class_weight': [None, 'balanced']  # Handling class imbalance
}

In [25]:
# Initialize Logistic Regression
log_reg = LogisticRegression()

# Perform Randomized Search with Cross Validation
random_search = RandomizedSearchCV(
    log_reg, param_distributions=param_dist,
    n_iter=20,  # Number of random searches (reduce to make it even faster)
    cv=3,  # Fewer folds for faster computation
    scoring='accuracy',
    n_jobs=-1,  # Use all CPU cores
    verbose=1,
    random_state=42
)
random_search.fit(X_train, y_train)

# Get best parameters
best_params = random_search.best_params_
print(f"Best Parameters: {best_params}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.11/dist-packages/scipy/optimize/_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(


Best Parameters: {'C': np.float64(1.2303823484477883), 'class_weight': None, 'max_iter': 3000, 'solver': 'newton-cg'}


In [26]:
# Train the optimized model with best parameters
best_log_reg = LogisticRegression(**best_params)
best_log_reg.fit(X_train, y_train)

# Generate Predictions
y_train_pred = best_log_reg.predict(X_train)
y_test_pred = best_log_reg.predict(X_test)

# Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

# Display Results
print(f"\nTraining Accuracy: {train_accuracy:.4f}")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

/usr/local/lib/python3.11/dist-packages/scipy/optimize/_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(



Training Accuracy: 0.6937
Training Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.61      0.66     24329
           1       0.69      0.77      0.72     26826

    accuracy                           0.69     51155
   macro avg       0.70      0.69      0.69     51155
weighted avg       0.69      0.69      0.69     51155

Training Confusion Matrix:
 [[14926  9403]
 [ 6265 20561]]

Testing Accuracy: 0.6959
Testing Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.61      0.66      6059
           1       0.69      0.77      0.73      6730

    accuracy                           0.70     12789
   macro avg       0.70      0.69      0.69     12789
weighted avg       0.70      0.70      0.69     12789

Testing Confusion Matrix:
 [[3714 2345]
 [1544 5186]]


**Random Forest**

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [28]:
#Train Random Forest Model
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [29]:
#Generate Predictions
y_train_pred = rf.predict(X_train)  # Predictions for training data
y_test_pred = rf.predict(X_test)    # Predictions for testing data


In [30]:
#Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

In [31]:
#Display Results
print(f"Training Accuracy: {train_accuracy:.4f}\n")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}\n")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

Training Accuracy: 1.0000

Training Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     24329
           1       1.00      1.00      1.00     26826

    accuracy                           1.00     51155
   macro avg       1.00      1.00      1.00     51155
weighted avg       1.00      1.00      1.00     51155

Training Confusion Matrix:
 [[24329     0]
 [    0 26826]]

Testing Accuracy: 0.9994

Testing Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6059
           1       1.00      1.00      1.00      6730

    accuracy                           1.00     12789
   macro avg       1.00      1.00      1.00     12789
weighted avg       1.00      1.00      1.00     12789

Testing Confusion Matrix:
 [[6055    4]
 [   4 6726]]


In [32]:
#hyperparameter tuning
rf = RandomForestClassifier(
    n_estimators=100,           # Fewer trees to reduce complexity
    max_depth=10,               # Control tree depth
    min_samples_split=5,        # Minimum samples needed to split a node
    min_samples_leaf=2,         # Minimum samples needed in a leaf node
    max_features='sqrt',        # Reduce the number of features considered at each split
    random_state=42
)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       random_state=42)

In [33]:
y_train_pred = rf.predict(X_train)  # Predictions for training data
y_test_pred = rf.predict(X_test)    # Predictions for testing data


In [34]:
#Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

In [35]:
#Display Results
print(f"Training Accuracy: {train_accuracy:.4f}\n")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}\n")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

Training Accuracy: 0.9516

Training Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.93      0.95     24329
           1       0.94      0.97      0.95     26826

    accuracy                           0.95     51155
   macro avg       0.95      0.95      0.95     51155
weighted avg       0.95      0.95      0.95     51155

Training Confusion Matrix:
 [[22581  1748]
 [  727 26099]]

Testing Accuracy: 0.9493

Testing Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.93      0.95      6059
           1       0.94      0.97      0.95      6730

    accuracy                           0.95     12789
   macro avg       0.95      0.95      0.95     12789
weighted avg       0.95      0.95      0.95     12789

Testing Confusion Matrix:
 [[5620  439]
 [ 209 6521]]


**XG BOOST**

In [36]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [37]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train, y_train)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:32:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [38]:
y_train_pred = xgb.predict(X_train)  # Predictions for training data
y_test_pred = xgb.predict(X_test)    # Predictions for testing data


In [39]:
#Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

In [40]:
#Display Results
print(f"Training Accuracy: {train_accuracy:.4f}\n")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}\n")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

Training Accuracy: 1.0000

Training Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     24329
           1       1.00      1.00      1.00     26826

    accuracy                           1.00     51155
   macro avg       1.00      1.00      1.00     51155
weighted avg       1.00      1.00      1.00     51155

Training Confusion Matrix:
 [[24329     0]
 [    0 26826]]

Testing Accuracy: 1.0000

Testing Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6059
           1       1.00      1.00      1.00      6730

    accuracy                           1.00     12789
   macro avg       1.00      1.00      1.00     12789
weighted avg       1.00      1.00      1.00     12789

Testing Confusion Matrix:
 [[6059    0]
 [   0 6730]]


In [41]:
#hyperparameter tuning
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [11:32:48] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [42]:
y_train_pred = xgb.predict(X_train)  # Predictions for training data
y_test_pred = xgb.predict(X_test)    # Predictions for testing data

In [43]:
#Evaluate Model Performance
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

train_class_report = classification_report(y_train, y_train_pred)
test_class_report = classification_report(y_test, y_test_pred)

train_conf_matrix = confusion_matrix(y_train, y_train_pred)
test_conf_matrix = confusion_matrix(y_test, y_test_pred)

In [44]:
#Display Results
print(f"Training Accuracy: {train_accuracy:.4f}\n")
print("Training Classification Report:\n", train_class_report)
print("Training Confusion Matrix:\n", train_conf_matrix)

print(f"\nTesting Accuracy: {test_accuracy:.4f}\n")
print("Testing Classification Report:\n", test_class_report)
print("Testing Confusion Matrix:\n", test_conf_matrix)

Training Accuracy: 0.9910

Training Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     24329
           1       0.99      0.99      0.99     26826

    accuracy                           0.99     51155
   macro avg       0.99      0.99      0.99     51155
weighted avg       0.99      0.99      0.99     51155

Training Confusion Matrix:
 [[24068   261]
 [  201 26625]]

Testing Accuracy: 0.9909

Testing Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      6059
           1       0.99      0.99      0.99      6730

    accuracy                           0.99     12789
   macro avg       0.99      0.99      0.99     12789
weighted avg       0.99      0.99      0.99     12789

Testing Confusion Matrix:
 [[6003   56]
 [  61 6669]]


In [49]:
#save the model
with open("randomforest_classifier.pkl", "wb") as f:
    pickle.dump(rf, f)

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [46]:
# Load Model & Encoders
try:
    rf_model = joblib.load('randomforest_model.pkl')
    batting_encoder = joblib.load('batting_encoder.pkl')
    bowling_encoder = joblib.load('bowling_encoder.pkl')
    venue_encoder = joblib.load('venue_encoder.pkl')
except FileNotFoundError as e:
    print(f"Error: {e}")
    exit()

# Input Data
input_data = {
    'match_id': [12345], 'inning': [2], 'batting_team': [4], 'bowling_team': [1], 'venue_canonical': [0],
    'cum_runs': [75], 'cum_wickets': [3], 'current_run_rate': [7.5], 'required_run_rate': [8.2], 'target': [180]
}

input_df = pd.DataFrame(input_data)

# Team & Venue Mapping (for readability)
team_mapping = {1: "Chennai Super Kings", 4: "Mumbai Indians"}
venue_mapping = {0: "Wankhede Stadium"}

batting_team = team_mapping.get(input_data['batting_team'][0], "Unknown")
bowling_team = team_mapping.get(input_data['bowling_team'][0], "Unknown")
venue = venue_mapping.get(input_data['venue_canonical'][0], "Unknown")

current_run_rate = input_data['current_run_rate'][0]
required_run_rate = input_data['required_run_rate'][0]

print(f"Batting: {batting_team}, Bowling: {bowling_team}, Venue: {venue}")
print(f"Current Run Rate: {current_run_rate:.2f}, Required Run Rate: {required_run_rate:.2f}")

# Ensure correct feature order
feature_order = ['match_id', 'inning', 'cum_runs', 'cum_wickets', 'current_run_rate',
                'required_run_rate', 'target', 'batting_team', 'bowling_team', 'venue_canonical']
input_features = input_df[feature_order]

# Make Prediction
try:
    # Get predicted outcome and probabilities
    predicted_win = rf_model.predict(input_features)[0]
    win_loss_probs = rf_model.predict_proba(input_features)[0] if hasattr(rf_model, 'predict_proba') else [0.5, 0.5]

    # Extract probabilities for win and loss
    batting_team_win_prob = win_loss_probs[1]  # Assuming 1 is the win class
    batting_team_loss_prob = win_loss_probs[0]  # Assuming 0 is the loss class

    # Determine the result
    result = "Win" if predicted_win == 1 else "Loss"

    # Print detailed prediction results
    print("\n--- MATCH PREDICTION SUMMARY ---")
    print(f"Current Situation: {batting_team} needs {input_data['target'][0] - input_data['cum_runs'][0]} runs with {10 - input_data['cum_wickets'][0]} wickets remaining")
    print(f"Overs played: {input_data['cum_runs'][0]/current_run_rate:.1f}")

    print("\n--- PREDICTION OUTCOME ---")
    print(f"{batting_team} (Batting): Win probability: {batting_team_win_prob:.2%}, Loss probability: {batting_team_loss_prob:.2%}")
    print(f"{bowling_team} (Bowling): Win probability: {batting_team_loss_prob:.2%}, Loss probability: {batting_team_win_prob:.2%}")

    print(f"\nOVERALL PREDICTION: {batting_team} will {'win' if predicted_win == 1 else 'lose'} the match")

    # Run rate analysis
    if required_run_rate > current_run_rate:
        run_rate_diff = required_run_rate - current_run_rate
        print(f"\nRUN RATE ANALYSIS: {batting_team} needs to increase their run rate by {run_rate_diff:.2f} to win")
    else:
        run_rate_diff = current_run_rate - required_run_rate
        print(f"\nRUN RATE ANALYSIS: {batting_team} is ahead of the required run rate by {run_rate_diff:.2f}")

except Exception as e:
    print(f"Prediction Error: {e}")

Batting: Mumbai Indians, Bowling: Chennai Super Kings, Venue: Wankhede Stadium
Current Run Rate: 7.50, Required Run Rate: 8.20

--- MATCH PREDICTION SUMMARY ---
Current Situation: Mumbai Indians needs 105 runs with 7 wickets remaining
Overs played: 10.0

--- PREDICTION OUTCOME ---
Mumbai Indians (Batting): Win probability: 43.52%, Loss probability: 56.48%
Chennai Super Kings (Bowling): Win probability: 56.48%, Loss probability: 43.52%

OVERALL PREDICTION: Mumbai Indians will lose the match

RUN RATE ANALYSIS: Mumbai Indians needs to increase their run rate by 0.70 to win
